# Day 4: Reservoir volumes, units, and text

**Time:** about 1 hour  |  **Needs:** Days 1 to 3
**Reservoir idea:** how much oil is in the ground? Bulk volume, pore volume, original oil in place (OOIP), and the oilfield units that make the answer come out right
**Python you will learn:** functions with several parameters (keyword arguments and defaults), named constants, strings (indexing, slicing, methods), parsing text into numbers, formatting output

By the end of today you will be able to:

1. calculate bulk volume, pore volume and OOIP of a reservoir, in the right units,
2. write functions with several parameters, call them by keyword, and give a parameter a default value,
3. use the string tools `len`, slices, `in`, `.strip()`, `.split()`, `.find()`, `.replace()`, `.startswith()`,
4. turn lines of text such as `DX 100 75 50 75 100` or `PERMX 150 md` into numbers,
5. format numbers neatly in printed output.

> **Companion reading (optional, about 40 minutes).** Charles Severance, *Python for Everybody* (free online: <https://www.py4e.com/html3/>), Chapter 6 "Strings": "A string is a sequence", "Getting the length of a string using len", "String slices", "Strings are immutable", "Looping and counting", "The in operator", "String comparison", "String methods", "Parsing strings", "Formatted String Literals".


## 0. Setup and recap

Run the next cell.

**Recap of Day 3.** `while` repeats until a condition fails; counters and accumulators are updated with `+=`; `for i in range(len(x))` gives you the position `i`; `break` leaves a loop.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("ready")

## 1. Story: the factor of 5.6

In 1999 NASA's Mars Climate Orbiter was lost. The investigation reported that one piece of software produced results in imperial units (pound-force seconds) while another expected metric units (newton seconds). Nothing was wrong with either program on its own. The mistake lived in the *units*.

Reservoir engineering invites the same mistake. A volume of 1000 ft³ and a volume of 1000 barrels differ by a factor of 5.615. In this very project the original FORTRAN program was found to be missing that 5.615 factor in its well terms (`docs/porting-notes.md`, decision 1), which made its wells 5.6 times too weak. Today's rule: **always know the unit of every number, and write it next to the variable name or in a comment.**

## 2. How much oil is in the reservoir?

A reservoir is rock with small pores that hold fluids. Four ideas:

- **Bulk volume** $V_b$: the volume of the whole rock, pores included. For a block, $V_b = \Delta x\,\Delta y\,\Delta z$.
- **Porosity** $\phi$: the fraction of the bulk volume that is pore space (0.20 means 20 %).
- **Pore volume** $V_p = \phi\,V_b$.
- **Saturation** $S$: the fraction of the pore space filled by a fluid. Water saturation $S_w$ and oil saturation $S_o = 1 - S_w$ (in an oil-water system).

The **original oil in place** is the oil that fills the pore space, expressed at surface conditions (stock-tank barrels, STB). Oil shrinks as it comes to the surface, which is accounted for by the **formation volume factor** $B_o$ (reservoir barrels per stock-tank barrel, rb/STB, about 1.5 here):

$$\text{OOIP} = \frac{V_p\,(1 - S_w)}{B_o} \quad\text{(with volumes in barrels)} \tag{1}$$

### Oilfield units

| Quantity | Unit | Remarks |
|---|---|---|
| length | ft | |
| area | acre | 1 acre = 43 560 ft² |
| volume (rock) | ft³ | |
| volume (fluid) | bbl | 1 bbl = 5.615 ft³ |
| permeability | md | millidarcy |
| pressure | psi | 1 psi = 6.894757 kPa |
| viscosity | cp | centipoise |
| rate | STB/day | surface barrels |
| $B_o$ | rb/STB | reservoir barrels per stock-tank barrel |

Since $V_p$ is in ft³ and OOIP is wanted in barrels, equation (1) needs the conversion factor:

$$\text{OOIP (STB)} = \frac{V_p\,[\text{ft}^3]\,(1 - S_w)}{5.615\;B_o} \tag{2}$$

If the area $A$ is in acres and the thickness $h$ in ft, there is a well-known shortcut: $1\ \text{acre-ft} = 43\,560 / 5.615 = 7758$ bbl, so

$$\text{OOIP (STB)} = \frac{7758\,A\,h\,\phi\,(1 - S_w)}{B_o} \tag{3}$$

### Named constants

A number such as 5.615 should not be typed into formulas again and again. Give it a name. By convention, constants are written in CAPITAL LETTERS and defined once at the top:

In [ ]:
BBL_TO_FT3 = 5.615        # ft3 per barrel
FT2_PER_ACRE = 43560      # ft2 per acre
PSI_TO_KPA = 6.894757     # kPa per psi

# 7758 bbl per acre-ft, calculated rather than typed:
print(FT2_PER_ACRE / BBL_TO_FT3)

## 3. Functions with several parameters

On Day 2 you wrote functions with one or two parameters. Real calculations need more. Here is the bulk volume of a block:

In [ ]:
def bulk_volume(dx, dy, dz):
    """Bulk volume (ft3) of a block with sides dx, dy, dz in ft."""
    return dx * dy * dz

print(bulk_volume(100, 100, 20))                 # by position
print(bulk_volume(dz=20, dx=100, dy=100))        # by name: the order does not matter

You can pass arguments **by position** (the first value goes to the first parameter) or **by name** (`dz=20`). Names make long calls readable and prevent mixing up values: with `ooip_stb(14.4e6, 0.2, 0.2, 1.5)` you must remember the order, but `ooip_stb(bulk_volume=14.4e6, phi=0.2, sw=0.2, bo=1.5)` explains itself.

A parameter can also have a **default value**, used when the caller does not give one:

```python
def ooip_stb(bulk_volume, phi, sw, bo, ft3_per_bbl=BBL_TO_FT3):
```
Here `ft3_per_bbl` can be left out: it is then 5.615. Parameters with defaults must come after the ones without.

**Your turn 3.1.** Write `pore_volume(bulk_volume, phi)` (returns $\phi V_b$) and `ooip_stb(bulk_volume, phi, sw, bo, ft3_per_bbl=BBL_TO_FT3)` which returns the OOIP in STB from equation (2). Use your `pore_volume` function inside `ooip_stb`.

In [ ]:
def pore_volume(bulk_volume, phi):
    """Pore volume, same unit as bulk_volume."""
    # YOUR CODE HERE
    pass


def ooip_stb(bulk_volume, phi, sw, bo, ft3_per_bbl=BBL_TO_FT3):
    """Original oil in place (STB) from bulk volume (ft3), porosity, water saturation and Bo (rb/STB)."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert pore_volume(1000, 0.2) == 200, "pore_volume(1000, 0.2) should be 200"
n = ooip_stb(bulk_volume=14.4e6, phi=0.2, sw=0.2, bo=1.5)
assert abs(n - 273553) < 2, f"ooip_stb for 14.4e6 ft3, phi 0.2, Sw 0.2, Bo 1.5 should be about 273553 STB, got {n:.0f}"
assert abs(ooip_stb(14.4e6, 0.2, 0.2, 1.5) - n) < 1e-6, "calling by position must give the same answer"
print(f"Correct: {n:,.0f} STB")

<details><summary>Solution</summary>

```python
def pore_volume(bulk_volume, phi):
    """Pore volume, same unit as bulk_volume."""
    return phi * bulk_volume


def ooip_stb(bulk_volume, phi, sw, bo, ft3_per_bbl=BBL_TO_FT3):
    """Original oil in place (STB) from bulk volume (ft3), porosity, water saturation and Bo (rb/STB)."""
    return pore_volume(bulk_volume, phi) * (1 - sw) / (ft3_per_bbl * bo)
```
</details>

(The number 14.4 million ft³ is not random: it is the bulk volume of the reservoir used in the thesis, as you will compute in Section 5.)

### What if the 5.615 is forgotten?

Predict the ratio between the OOIP calculated *without* the conversion (use `ft3_per_bbl=1`) and the correct one. Then run.

In [ ]:
right = ooip_stb(14.4e6, 0.2, 0.2, 1.5)
wrong = ooip_stb(14.4e6, 0.2, 0.2, 1.5, ft3_per_bbl=1)     # forgot ft3 -> bbl
print(f"right = {right:,.0f}   wrong = {wrong:,.0f}   ratio = {wrong / right:.3f}")

The ratio is exactly 5.615: the answer is 5.6 times too large, with no error message. Nothing in Python can warn you; only careful units can.

**Your turn 3.2.** Write `ooip_from_acres(area_acres, thickness_ft, phi, sw, bo)` using the shortcut equation (3) with $7758 = 43560/5.615$ (use your named constants, do not type 7758). The result should agree with `ooip_stb` for the same reservoir.

In [ ]:
def ooip_from_acres(area_acres, thickness_ft, phi, sw, bo):
    """OOIP (STB) from area in acres and thickness in ft, equation (3)."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
area_acres = 400 * 400 / FT2_PER_ACRE          # the thesis reservoir: 400 ft x 400 ft
a = ooip_from_acres(area_acres, 90, 0.2, 0.2, 1.5)
b = ooip_stb(400 * 400 * 90, 0.2, 0.2, 1.5)
assert abs(a - b) / b < 1e-6, f"the two methods should agree: {a:,.1f} vs {b:,.1f}. Is your 7758 equal to FT2_PER_ACRE / BBL_TO_FT3?"
print(f"Correct: {a:,.0f} STB by both methods")

<details><summary>Solution</summary>

```python
def ooip_from_acres(area_acres, thickness_ft, phi, sw, bo):
    """OOIP (STB) from area in acres and thickness in ft, equation (3)."""
    bbl_per_acre_ft = FT2_PER_ACRE / BBL_TO_FT3
    return bbl_per_acre_ft * area_acres * thickness_ft * phi * (1 - sw) / bo
```
</details>

**Your turn 3.3.** Pressures in this course are in psi, but papers in SI units use kPa. Write `psi_to_kpa(p)` and `kpa_to_psi(p)`, using `PSI_TO_KPA`.

In [ ]:
def psi_to_kpa(p):
    """Convert a pressure from psi to kPa."""
    # YOUR CODE HERE
    pass


def kpa_to_psi(p):
    """Convert a pressure from kPa to psi."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert abs(psi_to_kpa(1000) - 6894.757) < 1e-6, f"1000 psi should be 6894.757 kPa, got {psi_to_kpa(1000)}"
assert abs(kpa_to_psi(psi_to_kpa(4800)) - 4800) < 1e-9, "converting there and back must give the starting value"
print("Correct")

<details><summary>Solution</summary>

```python
def psi_to_kpa(p):
    """Convert a pressure from psi to kPa."""
    return p * PSI_TO_KPA


def kpa_to_psi(p):
    """Convert a pressure from kPa to psi."""
    return p / PSI_TO_KPA
```
</details>

## 4. Strings: text as a sequence

Text in Python is a **string**. You have been using strings since Day 1 (`"hello"`). A string is a *sequence of characters*, and it behaves like the lists you know: it has a length, positions (indexes) from 0, negative indexes from the end, and slices.

```
text = " P E R M X "     (without the spaces)
index:   0 1 2 3 4
```

Predict each answer, then run.

In [ ]:
text = "PERMX"

print(len(text))          # number of characters
print(text[0])            # first character
print(text[-1])           # last character
print(text[1:3])          # characters at positions 1 and 2
print(text + "_md")       # + joins strings
print("ER" in text)       # is one string inside another?
print(text == "PERMX", text == "permx")    # comparison is case sensitive

Two differences from lists:

1. Strings are **immutable**: you cannot change a character in place. `text[0] = "p"` gives a `TypeError`. You make a *new* string instead: `text = text.lower()`.
2. Quotes: `"..."` and `'...'` both work, but the two ends must match.

You can loop over a string just like over a list, one character at a time. This counts the letters `P` in a text:

In [ ]:
word = "PRODUCER P1 AND P2"
count = 0
for letter in word:
    if letter == "P":
        count += 1
print(count)

### String methods

A **method** is a function that belongs to a value; you call it with a dot: `text.upper()`. The ones you will use most:

| Method | Does | Example | Result |
|---|---|---|---|
| `.upper()`, `.lower()` | change case | `"Sw".upper()` | `"SW"` |
| `.strip()` | remove spaces at both ends | `"  PHI 0.2 ".strip()` | `"PHI 0.2"` |
| `.replace(a, b)` | replace text | `"1,500".replace(",", "")` | `"1500"` |
| `.startswith(s)` | begins with? | `"P1".startswith("P")` | `True` |
| `.find(s)` | position of the first match (-1 if none) | `"ab:cd".find(":")` | `2` |
| `.count(s)` | how many times | `"banana".count("a")` | `3` |
| `.split()` | cut into a list of words | `"DX 100 75".split()` | `["DX", "100", "75"]` |
| `.splitlines()` | cut into a list of lines | `"a\nb".splitlines()` | `["a", "b"]` |

`.split()` with no arguments cuts at any run of spaces; `.split(",")` cuts at commas. Run this and compare with the table:

In [ ]:
line = "  DX 100 75 50 75 100  "
print(line.strip())
print(line.split())
print(line.strip().startswith("DX"))
print("1,500".replace(",", ""))
print("ab:cd".find(":"))
print("banana".count("a"))

**Your turn 4.1.** The text `header = "RESERVOIR_NAME:  Thesis3D "` has a name after the colon. Using `.find(":")`, a slice, and `.strip()`, store in `name` just the name (`"Thesis3D"`, without spaces).

In [ ]:
header = "RESERVOIR_NAME:  Thesis3D "

# YOUR CODE HERE
name = None

In [ ]:
# Check (do not edit)
assert name == "Thesis3D", f"name should be 'Thesis3D', got {name!r}"
print("Correct")

<details><summary>Solution</summary>

```python
colon = header.find(":")
name = header[colon + 1:].strip()
```
(`header[colon + 1:]` is a slice with no end: "from after the colon to the end".)
</details>

**Your turn 4.2.** Well names are collected in one text: `names_text = "P1 P2 I1 P3 I2"`. Producers start with `P`, injectors with `I`. Split the text into a list, then use a loop and `.startswith` to count them: `n_producers` and `n_injectors`.

In [ ]:
names_text = "P1 P2 I1 P3 I2"

n_producers = 0
n_injectors = 0
# YOUR CODE HERE

In [ ]:
# Check (do not edit)
assert n_producers == 3, f"n_producers should be 3, got {n_producers}"
assert n_injectors == 2, f"n_injectors should be 2, got {n_injectors}"
print("Correct")

<details><summary>Solution</summary>

```python
for well in names_text.split():
    if well.startswith("P"):
        n_producers += 1
    elif well.startswith("I"):
        n_injectors += 1
```
</details>

## 5. From text to numbers: parsing

Simulators read their input from text files. The reservoir used in the thesis (Application 1, the same one the `kusim` examples use) is a 5 x 5 x 3 block model. A cleaned-up piece of its description looks like this (it is a string here; on Day 5 you will read such a thing from a file):

In [ ]:
input_text = '''
NX 5
NY 5
NZ 3
DX 100 75 50 75 100
DY 100 75 50 75 100
DZ 20 30 40
PHI 0.20
SWC 0.20
BO 1.5
'''

print(len(input_text.splitlines()), "lines (the first one is empty)")

Each line is a **keyword** followed by one or more numbers. To use them we must cut a line into words and convert the number words with `float()`. Everything read from text is a string until you convert it: `"100"` is text, `100.0` is a number you can calculate with.

**Your turn 5.1.** Write `parse_numbers(line)`. It splits a line into words, ignores the first word (the keyword), converts the rest to floats and returns them as a list. `parse_numbers("DX 100 75 50")` gives `[100.0, 75.0, 50.0]`.

In [ ]:
def parse_numbers(line):
    """Numbers after the keyword on a line, as a list of floats."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert parse_numbers("DX 100 75 50") == [100.0, 75.0, 50.0], f"got {parse_numbers('DX 100 75 50')}"
assert parse_numbers("PHI 0.20") == [0.2], "one number should give a list with one item"
assert parse_numbers("  DZ 20 30 40  ") == [20.0, 30.0, 40.0], "spaces around the line must not matter (split() handles them)"
print("Correct")

<details><summary>Solution</summary>

```python
def parse_numbers(line):
    """Numbers after the keyword on a line, as a list of floats."""
    words = line.split()
    numbers = []
    for word in words[1:]:        # skip the keyword at position 0
        numbers.append(float(word))
    return numbers
```
</details>

Now use it on the whole text. This loop goes through the lines, finds the keyword (the first word), and stores the numbers under the right variable. Read it, then run it. (It uses your `parse_numbers`.)

In [ ]:
dx = dy = dz = None
phi = swc = bo = None

for line in input_text.splitlines():
    line = line.strip()
    if line == "":
        continue                       # skip empty lines
    keyword = line.split()[0]
    numbers = parse_numbers(line)
    if keyword == "DX":
        dx = numbers
    elif keyword == "DY":
        dy = numbers
    elif keyword == "DZ":
        dz = numbers
    elif keyword == "PHI":
        phi = numbers[0]               # a single number: take it out of the list
    elif keyword == "SWC":
        swc = numbers[0]
    elif keyword == "BO":
        bo = numbers[0]

print("dx =", dx)
print("dy =", dy)
print("dz =", dz)
print("phi =", phi, " swc =", swc, " bo =", bo)

**Your turn 5.2.** The reservoir is a box whose sides are the sums of the block sizes. Store in `bulk_total` its bulk volume (ft³): (sum of `dx`) x (sum of `dy`) x (sum of `dz`). Then store in `ooip_total` the OOIP in STB, using your `ooip_stb` with the porosity, connate water saturation `swc` (use it as $S_w$) and `bo` that you parsed.

In [ ]:
# YOUR CODE HERE
bulk_total = None
ooip_total = None

In [ ]:
# Check (do not edit)
assert bulk_total == 14400000, f"bulk_total should be 14,400,000 ft3 (400 x 400 x 90), got {bulk_total}"
assert abs(ooip_total - 273553) < 2, f"ooip_total should be about 273,553 STB, got {ooip_total}"
print(f"Correct: bulk {bulk_total:,.0f} ft3, OOIP {ooip_total:,.0f} STB")

<details><summary>Solution</summary>

```python
bulk_total = sum(dx) * sum(dy) * sum(dz)
ooip_total = ooip_stb(bulk_total, phi, swc, bo)
```
</details>

> **A caution.** This is a rough number. It assumes the whole reservoir has the *same* water saturation (the connate value, 0.2) and the same $B_o$. In the thesis the initial saturation depends on the height above the water-oil contact, and $B_o$ depends on pressure. The simulator computes these block by block; today's calculation is the quick "hand estimate" you use to check that a simulator's total makes sense.

**Your turn 5.3.** A property line can also carry a *unit*: `"PERMX 150 md"`. Write `parse_property(line)` that returns three things: the name (text), the value (float), and the unit (text, or `""` if the line has no unit). `name, value, unit = parse_property("PERMX 150 md")` gives `"PERMX"`, `150.0`, `"md"`.

In [ ]:
def parse_property(line):
    """Split a line such as 'PERMX 150 md' into (name, value, unit). The unit may be missing."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert parse_property("PERMX 150 md") == ("PERMX", 150.0, "md"), f"got {parse_property('PERMX 150 md')}"
assert parse_property("PHI 0.2") == ("PHI", 0.2, ""), "a line with no unit should give an empty string as the unit"
name, value, unit = parse_property("  DATUM_PRESSURE  4800  psi ")
assert (name, value, unit) == ("DATUM_PRESSURE", 4800.0, "psi"), "extra spaces must not matter"
print("Correct")

<details><summary>Solution</summary>

```python
def parse_property(line):
    """Split a line such as 'PERMX 150 md' into (name, value, unit). The unit may be missing."""
    words = line.split()
    name = words[0]
    value = float(words[1])
    if len(words) > 2:
        unit = words[2]
    else:
        unit = ""
    return name, value, unit
```
</details>

## 6. Printing results neatly

A result such as `273552.98308103` is hard to read. An f-string lets you control how a number is shown, by adding `:` and a format after the value:

| Format | Meaning | `x = 273552.983` gives |
|---|---|---|
| `{x:.0f}` | 0 decimals | `273553` |
| `{x:,.0f}` | thousands separators | `273,553` |
| `{x:.2f}` | 2 decimals | `273552.98` |
| `{x:.3e}` | scientific notation | `2.736e+05` |
| `{x:>12,.0f}` | right-aligned in 12 characters | `     273,553` |
| `{0.2:.1%}` | percent | `20.0%` |

(`.format()` does the same: `"{:,.0f}".format(x)`. You will see it in older code.)

A good way to see your reservoir is to break it down by layer. This prints a table of the three layers of the thesis reservoir. Read it, then run it:

In [ ]:
side_x = sum(dx)
side_y = sum(dy)

print(f"{'layer':>5} {'dz (ft)':>8} {'bulk (ft3)':>13} {'OOIP (STB)':>12} {'share':>7}")
for k in range(len(dz)):
    bulk_layer = bulk_volume(side_x, side_y, dz[k])
    ooip_layer = ooip_stb(bulk_layer, phi, swc, bo)
    print(f"{k + 1:>5} {dz[k]:>8.0f} {bulk_layer:>13,.0f} {ooip_layer:>12,.0f} {ooip_layer / ooip_total:>7.1%}")
print(f"{'total':>5} {sum(dz):>8.0f} {bulk_total:>13,.0f} {ooip_total:>12,.0f} {1:>7.1%}")

Each layer holds oil in proportion to its thickness (22 %, 33 %, 44 %), since porosity, saturation and $B_o$ are the same in every layer.

**Your turn 6.1.** Write `summary_line(name, ooip)` that returns (not prints) a one-line text such as `"Thesis3D: 273,553 STB"`, with thousands separators and no decimals.

In [ ]:
def summary_line(name, ooip):
    """One-line summary of a reservoir's OOIP."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert summary_line("A", 1234567.8) == "A: 1,234,568 STB", f"got {summary_line('A', 1234567.8)!r}"
assert summary_line("Thesis3D", ooip_total) == "Thesis3D: 273,553 STB", f"got {summary_line('Thesis3D', ooip_total)!r}"
print("Correct")

<details><summary>Solution</summary>

```python
def summary_line(name, ooip):
    """One-line summary of a reservoir's OOIP."""
    return f"{name}: {ooip:,.0f} STB"
```
</details>

## 7. Bridge: the same ideas inside `kusim`

Open `src/kusim/constants.py`. It opens with the units of the whole simulator (ft, md, psi, cp, days, STB/day, ft³, rb/STB) and defines named constants:

```python
BBL_TO_FT3 = 5.615
DARCY = 0.006328        # = 0.001127 * 5.615
```

Look at the comment on `DARCY`: the Darcy-flow constant 0.001127 converts md·ft²·psi/(cp·ft) to bbl/day, and multiplying by 5.615 converts the barrels to ft³, so the simulator can work with cell volumes in ft³. That is today's lesson inside the simulator: *one named constant, one clearly stated unit system*.

Open `src/kusim/grid.py`, where the volume of every block is calculated with the same formula you wrote in `bulk_volume`:

```python
self.volume = self.dx * self.dy * self.dz
```
In the simulator `dx`, `dy` and `dz` are NumPy arrays with one value per block, so this one line computes the bulk volume of all blocks at once (Day 6 onwards). The pore volume of a block is then `phi * volume`, exactly your `pore_volume`.

And the input reader, `src/kusim/io.py`, does what you did in Section 5 (splitting text lines into numbers), with more care for the exact format of the legacy `INPUT.DAT`, where the labels after the numbers are ignored and the numbers are read by position.

## 8. Self-quiz

Write short answers as comments, then open the answers.

1. A reservoir has $V_b = 10^7$ ft³, $\phi=0.25$, $S_w=0.3$, $B_o=1.2$. Write (do not calculate) the Python line for OOIP in STB, using the constant `BBL_TO_FT3`.
2. What is the difference between `f(1, 2, 3)` and `f(a=1, b=2, c=3)`? Why prefer the second for long calls?
3. `s = "PERMX 150 md"`. What are `s[0]`, `s[-2:]`, `s.split()[1]` and `len(s)`?
4. Why can `s[0] = "p"` not work? What would you write to get `"pERMX"` from `"PERMX"`?
5. `float("150")` works. What does `float("150 md")` do, and how could you protect against it?

In [ ]:
# 1.
# 2.
# 3.
# 4.
# 5.

<details><summary>Answers</summary>

1. `ooip = 1e7 * 0.25 * (1 - 0.3) / (BBL_TO_FT3 * 1.2)`
2. The first gives values by position (order matters). The second names each parameter (order does not matter). Names make a call readable and prevent mixing up values.
3. `"P"`, `"md"`, `"150"` (a string, not a number!) and `12`.
4. Strings are immutable: they cannot be changed in place. Build a new string: `"p" + s[1:]`.
5. It raises a `ValueError` (the text is not a valid number). Wrap it in `try` / `except ValueError` (Day 2), or split the line first so that the number is a word on its own.
</details>

## 9. Pocket reference (today's Python)

| You want to... | Write |
|---|---|
| a constant | `BBL_TO_FT3 = 5.615` (capitals, defined once) |
| call by name | `f(bulk_volume=1e7, phi=0.25)` |
| default parameter | `def f(a, b=5):` |
| length, item, slice | `len(s)`, `s[0]`, `s[-1]`, `s[2:5]` |
| is it in? | `"ER" in s` |
| clean and cut | `s.strip()`, `s.split()`, `s.splitlines()` |
| search and change | `s.find(x)`, `s.replace(a, b)`, `s.count(x)` |
| starts with | `s.startswith("P")` |
| text to number | `float(word)`, `int(word)` |
| join strings | `a + b` |
| format a number | `f"{x:,.0f}"`, `f"{x:.2f}"`, `f"{x:.1%}"`, `f"{x:>10.2f}"` |
| return several values | `return name, value, unit` |

## 10. Wrap-up

Today you:

- calculated bulk volume, pore volume and OOIP, and saw how forgetting 5.615 gives a silently wrong answer,
- wrote functions with several parameters, keyword arguments and defaults, and named your constants,
- used strings like lists (indexes, slices, loops) and learned the main string methods,
- parsed lines of text into numbers and built a small input reader,
- printed neat tables with f-string formats,
- saw the same ideas in `kusim`'s `constants.py`, `grid.py` and `io.py`.

**Next, Day 5:** property tables for the fluids (PVT: $B_o$, viscosity against pressure). You will read a table from a file, store it as a list of lists, and write your own linear interpolation function, the same lookup that `kusim` uses.